# FFAI ResponseOptions API

This notebook introduces the `ResponseOptions` dataclass and the
`generate_response` signature introduced in the FFAI refactor.

Topics covered:

1. **New `generate_response` signature** -- `prompt`, `prompt_name`, `history`, and `options`
2. **`ResponseOptions` fields** -- `response_model`, `condition`, `response_format`, `strict`, and more
3. **Combining options** -- using multiple `ResponseOptions` fields together
4. **`history` via `ffai.history.raw`** -- accessing conversation history
5. **Provider kwargs** -- `temperature`, `max_tokens`, `tools` pass through as `**kwargs`

This notebook uses the **real Mistral API** via `FFMistralSmall`.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from pydantic import BaseModel, Field  # noqa: E402

from ffai.Clients.FFMistralSmall import FFMistralSmall  # noqa: E402
from ffai.core.response_options import ResponseOptions  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402

client = FFMistralSmall(
    max_tokens=1024,
    temperature=0.3,
)

ffai = FFAI(client)

print(f"FFAI initialized with model: {client.model}")
print(f"ResponseOptions fields: {', '.join(ResponseOptions.__dataclass_fields__.keys())}")

---

## 1. Basic Call (no options)

The simplest call passes just a prompt string. No `ResponseOptions` needed.

In [ ]:
result = ffai.workflow.generate_response(
    "Explain quantum entanglement in one sentence.",
    prompt_name="basic",
)

print(f"Response: {result.response}")
print(f"Model: {result.model}")
print(f"Status: {result.status}")

---

## 2. Structured Output with `response_model`

Pass a Pydantic model via `ResponseOptions(response_model=...)`. The LLM's
response is validated against the schema and returned as `result.parsed`.

In [ ]:
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: int = Field(ge=1, le=10, description="Rating out of 10")
    verdict: str = Field(description="One-word verdict: see, skip, or maybe")


result = ffai.workflow.generate_response(
    "Review the movie 'The Matrix'",
    prompt_name="movie_review",
    options=ResponseOptions(response_model=MovieReview),
)

print(f"Raw response:\n{result.response}\n")
print(f"Parsed: {result.parsed}")
print(f"Title: {result.parsed.title}")
print(f"Rating: {result.parsed.rating}/10")
print(f"Verdict: {result.parsed.verdict}")

---

## 3. Conditional Execution

Use `ResponseOptions(condition=...)` with a string expression to control
whether the LLM call executes. A falsy or `'False'` string skips the call.
The result's `status` field will be `'skipped'`.

In [ ]:
result_skip = ffai.workflow.generate_response(
    "Translate to French: hello",
    prompt_name="translate_skip",
    options=ResponseOptions(condition='False'),
)

print(f"Status: {result_skip.status}")
print(f"Response: {result_skip.response!r}")


result_run = ffai.workflow.generate_response(
    "Translate to French: hello",
    prompt_name="translate_run",
    options=ResponseOptions(condition='True'),
)

print(f"\nStatus: {result_run.status}")
print(f"Response: {result_run.response}")

---

## 4. JSON Mode with `response_format`

For raw JSON output without Pydantic validation, use
`ResponseOptions(response_format={'type': 'json_object'})`.
The `result.response` will contain the parsed JSON dict.

In [ ]:

result = ffai.workflow.generate_response(
    "List 3 colors as a JSON object with keys 'colors' (array of strings).",
    prompt_name="json_colors",
    options=ResponseOptions(response_format={"type": "json_object"}),
)

print(f"Response type: {type(result.response).__name__}")
print(f"Response: {result.response}")
print(f"Colors: {result.response['colors']}")

---

## 5. Accessing History via `ffai.history.raw`

Conversation history is tracked on the `ffai` instance. Each call to
`generate_response` appends to `ffai.history.raw`. You can inspect it as a
DataFrame or pass it as the `history=` top-level parameter for multi-turn.

In [ ]:
print(f'History entries so far: {len(ffai.history.raw)}')

for entry in ffai.history.raw:
    name = entry.get('prompt_name', '?')
    status = entry.get('status', '?')
    model = entry.get('model', '?')
    print(f'  {name:20s}  status={status:10s}  model={model}')

---

## 6. Multi-Turn with `history` Parameter

Pass `history=` (a list of dicts) to continue a conversation.
After each call, the updated history is in `ffai.history.raw`.

In [ ]:
# Reset history for a clean multi-turn demo
ffai.history.raw = []

result1 = ffai.workflow.generate_response(
    "My favorite color is blue. Remember that.",
    prompt_name="turn1",
    history=ffai.history.raw,
)
print(f"Turn 1: {result1.response}")

result2 = ffai.workflow.generate_response(
    "What is my favorite color?",
    prompt_name="turn2",
    history=ffai.history.raw,
)
print(f"Turn 2: {result2.response}")

---

## 7. Combining Options

`ResponseOptions` fields can be combined. Here we use `condition` +
`response_model` together.

In [ ]:
class Translation(BaseModel):
    original: str
    translated: str
    language: str


result = ffai.workflow.generate_response(
    "Translate 'goodbye' to Spanish",
    prompt_name="translation",
    options=ResponseOptions(
        response_model=Translation,
        condition='True',
    ),
)

print(f"Parsed: {result.parsed}")
print(f"{result.parsed.original} -> {result.parsed.translated} ({result.parsed.language})")

---

## 8. Provider Kwargs

Parameters like `temperature`, `max_tokens`, and `tools` are passed directly
as `**kwargs` to the provider. They are NOT inside `ResponseOptions`.

In [ ]:
result = ffai.workflow.generate_response(
    "Write a haiku about coding.",
    prompt_name="haiku",
    temperature=0.9,
    max_tokens=100,
)

print(f"Response:\n{result.response}")

---

## Summary

| Parameter | Where | Purpose |
|-----------|-------|---------|
| `prompt` | top-level positional | The user prompt string |
| `prompt_name` | top-level | Identifies the prompt for templating |
| `history` | top-level | Conversation history (list of message dicts) |
| `options` | top-level | `ResponseOptions` dataclass |
| `**kwargs` | top-level | Provider-specific: `temperature`, `max_tokens`, `tools`, etc. |

**`ResponseOptions` fields:**

| Field | Type | Purpose |
|-------|------|---------|
| `response_model` | `Type[BaseModel]` | Pydantic model for structured output |
| `condition` | `str` | String expression; call skips if falsy/False |
| `response_format` | `dict` | Raw JSON mode (`{"type": "json_object"}`) |
| `strict` | `bool` | Strict schema validation |
| `model` | `str` | Override the default model |
| `system_instructions` | `str` | Override system instructions |
| `abort_condition` | `str` | String expression; abort if truthy |

**`ResponseResult` key fields:**

| Field | Type | Purpose |
|-------|------|---------|
| `response` | `Any` | Raw response (str, dict, or parsed model) |
| `status` | `str` | `'success'`, `'skipped'`, or `'failed'` |
| `parsed` | `Any` | Validated Pydantic model instance |
| `model` | `str` | Model used for this response |
| `cost_usd` | `float` | Cost in USD |
| `duration_ms` | `float` | Wall time in milliseconds |